<a href="https://colab.research.google.com/github/Kavishka2401/CustomerChurnPredictionSystem/blob/master/Final_NN_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount the google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Core libraries
import numpy as np
import pandas as pd

In [3]:
# Provide path
df = pd.read_csv('/content/drive/MyDrive/processed_data_NN.csv')

In [4]:
# Copy data
churn_data = df.copy()

In [5]:
churn_data.columns

Index(['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'TotalCharges_numeric', 'TotalCharges_log', 'gender_Male',
       'SeniorCitizen_1', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='object')

In [6]:
churn_data.head()

,customerID,tenure,MonthlyCharges,TotalCharges,Churn,TotalCharges_numeric,TotalCharges_log,gender_Male,SeniorCitizen_1,Partner_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0.013889,0.115423,29.85,0,29.85,0.072892,0,0,1,...,0,0,0,0,0,0,1,0,1,0
1,5575-GNVDE,0.472222,0.385075,1889.5,0,1889.50,0.749358,1,0,0,...,0,0,0,0,1,0,0,0,0,1
2,3668-QPYBK,0.027778,0.354229,108.15,1,108.15,0.280590,1,0,0,...,0,0,0,0,0,0,1,0,0,1
3,7795-CFOCW,0.625000,0.239303,1840.75,0,1840.75,0.745063,1,0,0,...,0,0,0,0,1,0,0,0,0,0
4,9237-HQITU,0.027778,0.521891,151.65,1,151.65,0.335724,0,0,0,...,0,0,0,0,0,0,1,0,1,0


In [7]:
import pandas as pd

# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows if needed
pd.set_option('display.max_rows', None)

# Prevent truncation of long strings in cells
pd.set_option('display.max_colwidth', None)

churn_data.head()

,customerID,tenure,MonthlyCharges,TotalCharges,Churn,TotalCharges_numeric,TotalCharges_log,gender_Male,SeniorCitizen_1,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,0.013889,0.115423,29.85,0,29.85,0.072892,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0
1,5575-GNVDE,0.472222,0.385075,1889.5,0,1889.50,0.749358,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1
2,3668-QPYBK,0.027778,0.354229,108.15,1,108.15,0.280590,1,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1
3,7795-CFOCW,0.625000,0.239303,1840.75,0,1840.75,0.745063,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0
4,9237-HQITU,0.027778,0.521891,151.65,1,151.65,0.335724,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0


In [8]:
churn_data['NoInternetService'] = churn_data[[
    'OnlineSecurity_No internet service',
    'OnlineBackup_No internet service',
    'DeviceProtection_No internet service',
    'TechSupport_No internet service',
    'StreamingTV_No internet service',
    'StreamingMovies_No internet service'
]].any(axis=1).astype(int)

service_yes_cols = [
    'PhoneService_Yes',
    'MultipleLines_Yes',
    'OnlineSecurity_Yes',
    'OnlineBackup_Yes',
    'DeviceProtection_Yes',
    'TechSupport_Yes',
    'StreamingTV_Yes',
    'StreamingMovies_Yes'
]

churn_data['service_yes'] = churn_data[service_yes_cols].sum(axis=1)
churn_data['AvgChargePerMonth'] = churn_data['TotalCharges_numeric'] / (churn_data['tenure'] + 1)

drop_cols = ['customerID',
             'TotalCharges_numeric',
             'TotalCharges',
             # No internet columns
             'OnlineSecurity_No internet service',
             'OnlineBackup_No internet service',
             'DeviceProtection_No internet service',
             'TechSupport_No internet service',
             'StreamingTV_No internet service',
             'StreamingMovies_No internet service',
             # Service columns
             'PhoneService_Yes',
             'MultipleLines_Yes',
             'OnlineSecurity_Yes',
             'OnlineBackup_Yes',
             'DeviceProtection_Yes',
             'TechSupport_Yes',
             'StreamingTV_Yes',
             'StreamingMovies_Yes']

# Drop the columns from the DataFrame
churn_data = churn_data.drop(columns=drop_cols)

# Check the remaining columns
print(churn_data.columns)

Index(['tenure', 'MonthlyCharges', 'Churn', 'TotalCharges_log', 'gender_Male',
       'SeniorCitizen_1', 'Partner_Yes', 'Dependents_Yes',
       'MultipleLines_No phone service', 'InternetService_Fiber optic',
       'InternetService_No', 'Contract_One year', 'Contract_Two year',
       'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'NoInternetService', 'service_yes', 'AvgChargePerMonth'],
      dtype='object')


In [10]:
churn_data.head()

,tenure,MonthlyCharges,Churn,TotalCharges_log,gender_Male,SeniorCitizen_1,Partner_Yes,Dependents_Yes,MultipleLines_No phone service,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,NoInternetService,service_yes,AvgChargePerMonth
0,0.013889,0.115423,0,0.072892,0,0,1,0,1,0,0,0,0,1,0,1,0,0,1,29.441096
1,0.472222,0.385075,0,0.749358,1,0,0,0,0,0,0,1,0,0,0,0,1,0,3,1283.433962
2,0.027778,0.354229,1,0.280590,1,0,0,0,0,0,0,0,0,1,0,0,1,0,3,105.227027
3,0.625000,0.239303,0,0.745063,1,0,0,0,1,0,0,1,0,0,0,0,0,0,3,1132.769231
4,0.027778,0.521891,1,0.335724,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,147.551351


In [12]:
from sklearn.preprocessing import MinMaxScaler

# Select features to scale
num_cols = ['AvgChargePerMonth']

# Initialize scaler
scaler = MinMaxScaler()

# Fit and transform the numerical columns
churn_data[num_cols] = scaler.fit_transform(churn_data[num_cols])

# Check the first few rows
churn_data.head()

,tenure,MonthlyCharges,Churn,TotalCharges_log,gender_Male,SeniorCitizen_1,Partner_Yes,Dependents_Yes,MultipleLines_No phone service,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,NoInternetService,service_yes,AvgChargePerMonth
0,0.013889,0.115423,0,0.072892,0,0,1,0,1,0,0,0,0,1,0,1,0,0,1,0.002521
1,0.472222,0.385075,0,0.749358,1,0,0,0,0,0,0,1,0,0,0,0,1,0,3,0.292538
2,0.027778,0.354229,1,0.280590,1,0,0,0,0,0,0,0,0,1,0,0,1,0,3,0.020048
3,0.625000,0.239303,0,0.745063,1,0,0,0,1,0,0,1,0,0,0,0,0,0,3,0.257693
4,0.027778,0.521891,1,0.335724,0,0,0,0,0,1,0,0,0,1,0,1,0,0,1,0.029837


In [ ]:
from sklearn.model_selection import train_test_split

# Define X and y
X = churn_data.drop('Churn', axis=1)
y = churn_data['Churn']

# Randomly select 10 samples for inference
X_infer = X.sample(n=10, random_state=42)
y_infer = y.loc[X_infer.index]

# Drop the inference samples from the main dataset
X_rest = X.drop(X_infer.index)
y_rest = y.drop(y_infer.index)

# Split the rest into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_rest, y_rest,
    test_size=0.2,
    random_state=42,
    stratify=y_rest
)

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_infer:", X_infer.shape)